In [1]:
#This code takes the original data, the prediction of nns and the BMS, computes the rmse and mae and saves everything into a dataframe 

In [19]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure
import seaborn as sns
import matplotlib.gridspec as gridspec
import ast
import sys
sys.path.append('../no_degeneracy/')
sys.path.append('../no_degeneracy/Prior/')
from mcmc import *
from parallel import *
from fit_prior import read_prior_par
from sklearn.metrics import mean_squared_error
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import mean_absolute_error

In [20]:
def clean_index(dataframe):
    dataframe.set_index('Unnamed: 0', inplace=True)
    dataframe.index.name = None
    dataframe= dataframe.reset_index(drop=True)
    return dataframe

def add_bms_pred(dataframe, bms_trace, number_param):
    VARS = ['x1',]
    x = dn[[c for c in VARS]].copy()
    y=dataframe.noise

    if number_param==10:
        prior_par = read_prior_par('../no_degeneracy/Prior/final_prior_param_sq.named_equations.nv1.np10.2017-10-18 18:07:35.089658.dat')
    elif number_param==20:
        prior_par = read_prior_par('../no_degeneracy/Prior/final_prior_param_sq.named_equations.nv1.np20.maxs200.2024-05-10 162907.551306.dat')

    #mdl model
    minrow = bms_trace[bms_trace.H == min(bms_trace.H)].iloc[0]
    minH, minexpr, minparvals = minrow.H, minrow.expr, ast.literal_eval(minrow.parvals)

    t = Tree(
        variables=list(x.columns),
        parameters=['a%d' % i for i in range(number_param)],
        x=x, y=y,
        prior_par=prior_par,
        max_size=200,
        from_string=minexpr,
    )

    t.set_par_values(deepcopy(minparvals))

    dplot = deepcopy(dn)
    dplot['ybms'] = t.predict(x)

    return dplot
    

In [21]:
#Read NN and BMS data
functions=['leaky_ReLU', 'tanh'] #tanh, leaky_ReLU
realizations=2
N=9
sigmas=[0.0, 0.02, 0.04,0.06, 0.08, 0.10, 0.12, 0.14, 0.16, 0.18, 0.20]

resolution='4e-3x' #0.5x, 1x, 2x, 4e-3x
resolutions={'0.5x':'0.1', '1x':'0.05' , '2x': '0.025' , '4e-3x':'0.004' }

NPAR=10 #10, 20
steps=50000


rmse_nn_train=[];rmse_nn_test=[]
rmse_mdl_train=[];rmse_mdl_test=[]

mae_nn_train=[];mae_nn_test=[]
mae_mdl_train=[];mae_mdl_test=[]

n_index=[];r_index=[];sigma_index=[];function_index=[]

#Put mae and rmse of each simulation (on nn and bms) in a dataframe
for function in functions:

    for sigma in sigmas:

        for r in range(realizations+1):

            #Read data and nn predictions
            file_model='NN_no_overfit_' + function + '_sigma_' + str(sigma) + '_r_' + str(r) + '.csv'
            model_d='../../data/nns/' + resolution + '_resolution/approximation/' + file_model
            d=pd.read_csv(model_d)

            n_points=int(len(d.index)/10)
            train_fraction=3/4;train_size=int(n_points*train_fraction)

            for n in range(N+1):
                n_index.append(n);r_index.append(r);sigma_index.append(sigma);function_index.append(function)
            
                dn=d[d['rep']==n]
                dn=clean_index(dn)

                #Read BMS data
                filename='BMS_'+function+'_n_'+str(n)+'_sigma_'+str(sigma)+ '_r_' + str(r) + '_res_' + resolutions[resolution] + '_trace_'\
                +str(steps)+'_prior_'+str(NPAR)+ '.csv'

                print(function, sigma, n, r)
                
                try:
                    trace=pd.read_csv('../../data/MSTraces/' + resolution + '_resolution/' + filename, sep=';', header=None, names=['t', 'H', 'expr', 'parvals', 'kk1', 'kk2','kk3'])
                    dplot=add_bms_pred(dn, trace, NPAR)
                except FileNotFoundError:
                    dplot = deepcopy(dn) #If no bms errors available, fill the dataframe with zeros
                    dplot['ybms'] = [0] * len(dplot)
                

                #Compute errors
                #-----------------------------------------------------------------------------------------------------------------------
                #nns
                rmse_nn_train_i=root_mean_squared_error(dplot.loc[:train_size-1]['ymodel'],dplot.loc[:train_size -1]['y'])
                rmse_nn_train.append(rmse_nn_train_i)
            
                rmse_nn_test_i=root_mean_squared_error(dplot.loc[train_size-1:]['ymodel'],dplot.loc[train_size -1:]['y'])
                rmse_nn_test.append(rmse_nn_test_i)

                mae_nn_train_i=mean_absolute_error(dplot.loc[:train_size-1]['ymodel'],dplot.loc[:train_size -1]['y'])
                mae_nn_train.append(mae_nn_train_i)
            
                mae_nn_test_i=mean_absolute_error(dplot.loc[train_size-1:]['ymodel'],dplot.loc[train_size -1:]['y'])
                mae_nn_test.append(mae_nn_test_i)

                
                #bms
                try:
                    rmse_mdl_train_i=root_mean_squared_error(dplot.loc[:train_size-1]['ybms'],dn.loc[:train_size-1]['y'])
                except ValueError:
                    rmse_mdl_train_i=np.inf
                rmse_mdl_train.append(rmse_mdl_train_i)

                try:
                    rmse_mdl_test_i=root_mean_squared_error(dplot.loc[train_size-1:]['ybms'],dn.loc[train_size-1:]['y'])
                except (ValueError, RuntimeWarning) as e:
                    rmse_mdl_test_i=np.inf
                
                rmse_mdl_test.append(rmse_mdl_test_i)

                try:
                    mae_mdl_train_i=mean_absolute_error(dplot.loc[:train_size-1]['ybms'],dplot.loc[:train_size -1]['y'])
                except ValueError:
                    mae_mdl_train_i=np.inf
                mae_mdl_train.append(mae_mdl_train_i)

                try:
                    mae_mdl_test_i=mean_absolute_error(dplot.loc[train_size-1:]['ybms'],dplot.loc[train_size -1:]['y'])
                except ValueError:
                    mae_mdl_test_i=np.inf
                
                mae_mdl_test.append(mae_mdl_test_i)
                #-----------------------------------------------------------------------------------------------------------------------


#Save all in a dataframe
errors_df=pd.DataFrame({'sigma':sigma_index, 'function':function_index, 'mae_nn_train':mae_nn_train, 'mae_nn_test':mae_nn_test, 'mae_mdl_train':mae_mdl_train, 
                        'mae_mdl_test':mae_mdl_test, 'rmse_nn_train':rmse_nn_train, 'rmse_nn_test': rmse_nn_test, 
                        'rmse_mdl_train':rmse_mdl_train, 'rmse_mdl_test': rmse_mdl_test, 'n':n_index, 'r': r_index})
errors_df.to_csv('../../data/errors_median_' + resolution + '.csv')
display(errors_df)

leaky_ReLU 0.0 0 0


<lambdifygenerated-53647>:2: RuntimeWarning: invalid value encountered in power
  return abs(x1 + exp(x1**(2*x1)*tanh(_a2_*x1*cos((_a3_ + x1)*(_a0_/x1 + _a1_)) + _a4_ + (_a5_ + _a6_)*sin(_a0_*x1))**2/_a3_**2))
/export/home/shared/Projects/ANN/Sergio/BMS_approximator/bin/ann_functions/../no_degeneracy/mcmc.py:654: OptimizeWarning: Covariance of the parameters could not be estimated
  res = curve_fit(
<lambdifygenerated-53648>:2: RuntimeWarning: invalid value encountered in power
  return abs(x1 + exp(x1**(2*x1)*tanh(_a2_*x1*cos((_a3_ + x1)*(_a0_/x1 + _a1_)) + _a4_ + (_a5_ + _a6_)*sin(_a0_*x1))**2/_a3_**2))
<lambdifygenerated-53649>:2: RuntimeWarning: overflow encountered in exp
  return abs(x1 + exp(_a6_**(2*x1)*tanh(_a2_*x1*cos((_a3_ + x1)*(_a0_/x1 + _a1_)) + _a4_ + (_a5_ + _a6_)*sin(_a0_*x1))**2/_a3_**2))
/usr/local/lib/python3.10/dist-packages/scipy/optimize/_minpack_py.py:1017: RuntimeWarning: overflow encountered in square
  cost = np.sum(infodict['fvec'] ** 2)
/usr/local/lib/pytho

leaky_ReLU 0.0 1 0


<lambdifygenerated-53699>:2: RuntimeWarning: overflow encountered in cosh
  return x1 + (_a1_*(_a0_ + x1*(_a2_ + x1)) + x1 + cosh(_a6_*x1))**2
<lambdifygenerated-53699>:2: RuntimeWarning: overflow encountered in square
  return x1 + (_a1_*(_a0_ + x1*(_a2_ + x1)) + x1 + cosh(_a6_*x1))**2
/export/home/shared/Projects/ANN/Sergio/BMS_approximator/bin/ann_functions/../no_degeneracy/mcmc.py:654: OptimizeWarning: Covariance of the parameters could not be estimated
  res = curve_fit(
<lambdifygenerated-53723>:2: RuntimeWarning: overflow encountered in cosh
  return x1 + (_a3_*(_a1_*(_a0_ + x1*(_a2_ + x1)) + cosh(_a6_*x1))/cosh(x1*(_a4_*_a7_ + _a5_*x1)*cos(x1)) + x1)**2


leaky_ReLU 0.0 2 0


...

tanh 0.2 7 2


/export/home/shared/Projects/ANN/Sergio/BMS_approximator/bin/ann_functions/../no_degeneracy/mcmc.py:654: OptimizeWarning: Covariance of the parameters could not be estimated
  res = curve_fit(
<lambdifygenerated-75207>:2: RuntimeWarning: overflow encountered in exp
  return _a3_/(x1 + exp(x1**2*(_a1_ + x1)**2))
<lambdifygenerated-75208>:2: RuntimeWarning: overflow encountered in exp
  return _a3_/(x1 + exp(x1**2*(_a1_ + x1)**2))
<lambdifygenerated-75209>:2: RuntimeWarning: overflow encountered in exp
  return _a3_/(x1 + exp(x1**2*(_a1_ + x1)**2))
<lambdifygenerated-75210>:2: RuntimeWarning: overflow encountered in exp
  return _a3_/(x1 + exp(x1**2*(_a1_ + x1)**2))
<lambdifygenerated-75211>:2: RuntimeWarning: overflow encountered in exp
  return _a3_/(x1 + exp(x1**2*(_a1_ + x1)**2))
<lambdifygenerated-75212>:2: RuntimeWarning: overflow encountered in exp
  return _a3_/(x1 + exp(x1**2*(_a1_ + x1)**2))
<lambdifygenerated-75221>:2: RuntimeWarning: invalid value encountered in power
  retur

tanh 0.2 8 2
tanh 0.2 9 2


/export/home/shared/Projects/ANN/Sergio/BMS_approximator/bin/ann_functions/../no_degeneracy/mcmc.py:654: OptimizeWarning: Covariance of the parameters could not be estimated
  res = curve_fit(


,sigma,function,mae_nn_train,mae_nn_test,mae_mdl_train,mae_mdl_test,rmse_nn_train,rmse_nn_test,rmse_mdl_train,rmse_mdl_test,n,r
0,0.0,leaky_ReLU,0.001531,0.040837,0.000496,0.021230,0.001936,0.049012,0.000634,0.029018,0,0
1,0.0,leaky_ReLU,0.002255,0.155396,0.001845,1.389800,0.003058,0.164442,0.002424,2.676518,1,0
2,0.0,leaky_ReLU,0.003467,0.036872,0.002618,0.068700,0.005003,0.043094,0.004424,0.080606,2,0
3,0.0,leaky_ReLU,0.001850,0.056598,0.000623,0.298642,0.002887,0.060429,0.000860,0.347085,3,0
4,0.0,leaky_ReLU,0.001471,0.235353,0.001347,inf,0.002086,0.279269,0.001780,inf,4,0
...,...,...,...,...,...,...,...,...,...,...,...,...
655,0.2,tanh,0.027551,0.067124,0.024260,0.117640,0.034761,0.083256,0.037562,0.123875,5,2
656,0.2,tanh,0.028916,0.120084,0.026511,0.384823,0.037839,0.138045,0.034651,0.451462,6,2
657,0.2,tanh,0.035341,0.063014,0.012995,0.077759,0.042194,0.078681,0.019436,0.091211,7,2
658,0.2,tanh,0.026682,0.505530,0.019877,0.094024,0.035372,0.565568,0.022791,0.096903,8,2
